# Cross-Session Behavioral Analysis Summary

This notebook loads behavioral results from multiple imaging sessions across animals and performs statistical comparisons. It calls many functions in xsession_utils.py as well as the session configuration file (session_config.py)

In [ ]:
# Imports
import pickle
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
from session_config import *
from xsession_utils import *

#### Define Session Metadata

In [ ]:
SESSION_REF

#### Load all behavioral data

In [ ]:
ANIMAL_PATHS

In [ ]:
print("Loading behavioral results...\n")
all_behavior_data = load_all_behavior_data(ANIMAL_PATHS)

In [ ]:
all_behavior_data

#### Define plotting Functions

In [ ]:
def plot_paired_comparison(df, session_indices, metric_name='', chance_line=None, 
                           title=None, figsize=(4, 5), save_path=None):
    """
    Create a paired comparison plot with individual animal lines.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Output from extract_metric()
    session_indices : list of 2 ints
        The two sessions to compare
    metric_name : str
        Name to display on y-axis
    chance_line : float or None
        If provided, draws a horizontal line at this value
    """
    pivot_df = pivot_for_comparison(df, session_indices)
    complete_df = pivot_df.dropna()
    
    col1, col2 = complete_df.columns
    colors = [SESSION_CONFIG['colors'][i] for i in session_indices]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot individual animal lines
    for animal in complete_df.index:
        ax.plot([0, 1], [complete_df.loc[animal, col1], complete_df.loc[animal, col2]], 
                'o-', color='gray', alpha=0.5, markersize=8, linewidth=1.5)
    
    # Plot means with error bars
    means = [complete_df[col1].mean(), complete_df[col2].mean()]
    sems = [complete_df[col1].sem(), complete_df[col2].sem()]
    
    ax.errorbar([0, 1], means, yerr=sems, fmt='s', markersize=12, 
                color='black', capsize=5, capthick=2, linewidth=2, zorder=10)
    
    # Chance line
    if chance_line is not None:
        ax.axhline(chance_line, color='red', linestyle='--', alpha=0.5, label=f'Chance ({chance_line})')
    
    # Formatting
    ax.set_xticks([0, 1])
    ax.set_xticklabels([col1, col2], fontsize=12)
    ax.set_xlim(-0.3, 1.3)
    
    if metric_name:
        ax.set_ylabel(metric_name, fontsize=12)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()

    if title is not None:
        plt.title(title)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    
    return fig, ax


def plot_metric_across_sessions(df, metric_name='', chance_line=None, 
                                 title=None, figsize=(8, 5), save_path=None):
    """
    Plot a metric across all sessions with individual animal trajectories.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Output from extract_metric()
    metric_name : str
        Name to display on y-axis
    chance_line : float or None
        If provided, draws a horizontal line at this value
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # Get unique sessions present in data
    session_indices = sorted(df['session_idx'].unique())
    x_positions = list(range(len(session_indices)))
    x_labels = [SESSION_CONFIG['short_labels'][i] for i in session_indices]
    colors = [SESSION_CONFIG['colors'][i] for i in session_indices]
    
    # Plot individual animal trajectories
    for animal in df['animal'].unique():
        animal_data = df[df['animal'] == animal].set_index('session_idx')
        y_values = [animal_data.loc[s, 'value'] if s in animal_data.index else np.nan 
                    for s in session_indices]
        ax.plot(x_positions, y_values, 'o-', color='gray', alpha=0.4, 
                markersize=6, linewidth=1, label=animal)
    
    # Plot means with error bars
    means = []
    sems = []
    for s in session_indices:
        session_values = df[df['session_idx'] == s]['value'].dropna()
        means.append(session_values.mean())
        sems.append(session_values.sem() if len(session_values) > 1 else 0)
    
    ax.errorbar(x_positions, means, yerr=sems, fmt='s-', markersize=10, 
                color='black', capsize=4, capthick=2, linewidth=2, zorder=10)
    
    # Color the x-axis tick labels
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels, fontsize=11)
    for ticklabel, color in zip(ax.get_xticklabels(), colors):
        ticklabel.set_color(color)
    
    # Chance line
    if chance_line is not None:
        ax.axhline(chance_line, color='red', linestyle='--', alpha=0.5)
    
    if metric_name:
        ax.set_ylabel(metric_name, fontsize=12)
    ax.set_xlabel('Session', fontsize=12)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    if title is not None:
        plt.title(title)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    
    return fig, ax

#### Flexible analysis with template

In [ ]:
###loop schema:
#1) extract metric for all sessions and all animals --> save as .csv
#2) plot the metric across all sessions and save
#3) compare each timepoint to null (if applicable) --> benjamini hochberg correction for multiple comparisons (do this comparison for all five timepoints and then separately for all four presep timepoints)
#4) pull out the following comparisons and run stats:
#4a) novel/novel ss -> novel familiar SS
#4b) novel/novel os -> novel familiar OS
#4c) novel/novel ss -> novel/novel OS
#4d) novel/familiar ss -> novel/familiar OS
#5) benjamini hochberg correction for five comparisons (and then four if excluding the long-term timepoint)
#6) plot and save each comparison

In [ ]:
res_path = os.path.join(CROSS_SESSION_RESULTS_DIR,'imaging_session_behavior_svgs')

In [ ]:

from statsmodels.stats.multitest import multipletests


# ── Constant: the 5 planned paired comparisons ──────────────────────────────
# Each entry: (SESSION_SUBSETS key, short name for filenames/columns, plot title)

PLANNED_COMPARISONS = [
    ('same_sex',       'same_sex',       'same-sex conspecifics (before and after cohousing)'),
    ('os_presep',      'os_presep',      'opposite-sex conspecifics (before and after cohousing)'),
    ('os_familiar',    'os_perisep',     'opposite-sex conspecifics (before and after separation)'),
    ('novel',          'novel_ss_vs_os', 'two novel conspecifics (same-sex vs. opposite-sex)'),
    ('familiar_presep','fam_ss_vs_os',   'one novel + one familiar (same-sex vs. opposite-sex)'),
]


def results_compilation_loop(all_behavior_data, metric, metric_label, res_path,
                             sessions=None, chance=None,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=True,
                             extractor=extract_metric):
    """
    Full analysis pipeline for one behavioral metric across sessions.

    Steps:
      1. Extract metric → tidy DataFrame, optionally save CSV
      2. Plot metric across sessions, optionally save figure
      3. If chance is not None: one-sample tests per session with BH-FDR
      4. Five planned paired comparisons with BH-FDR
      5. Plot each paired comparison, optionally save figures

    Parameters
    ----------
    all_behavior_data : dict
        Nested dict from load_all_behavior_data().
    metric : str
        Key in the per-session results dict (e.g. 's1_first_five_min_partner_preference_index').
    metric_label : str
        Human-readable label used for axis labels, file names, and subdirectory.
    res_path : str
        Parent results directory. A subdirectory named metric_label is created inside.
    sessions : list[int] or None
        Session indices to include (default: SESSION_SUBSETS['social']).
    chance : float or None
        Null value for one-sample tests. If None, one-sample tests are skipped.
    one_way_test : str
        'ttest' or 'wilcoxon' for one-sample tests.
    two_way_test : str
        'ttest' or 'wilcoxon' for paired comparisons.
    saving : bool
        Master switch — if False, nothing is saved to disk (but all analyses still run
        and are returned).
    extractor : callable
        Function with extract_metric's signature, used to build metric_df.
        Pass derived_extractor(func) to run a quantity computed from the stored
        keys rather than read from one (e.g. total interaction duration.

    Returns
    -------
    dict with keys:
        'metric_df'        : pd.DataFrame  — the extracted metric (tidy)
        'one_way_df'       : pd.DataFrame or None — one-sample results with FDR columns
        'comparisons_df'   : pd.DataFrame or None — paired comparison results with FDR columns
    """
    if sessions is None:
        sessions = SESSION_SUBSETS['social']

    # ── output directory ─────────────────────────────────────────────────
    metric_path = os.path.join(res_path, metric_label)
    if saving and not os.path.isdir(metric_path):
        os.makedirs(metric_path, exist_ok=True)

    # ── 1. extract metric ────────────────────────────────────────────────
    metric_df = extractor(all_behavior_data, metric,
                          session_indices=sessions, animals=None)

    if saving:
        metric_df.to_csv(os.path.join(metric_path, f"{metric_label}_summary.csv"),
                         index=False)

    # ── 2. cross-session trajectory plot ─────────────────────────────────
    fig_path = os.path.join(metric_path, f"{metric_label}_across_sessions.svg") if saving else None
    plot_metric_across_sessions(metric_df, metric_name=metric_label,
                                title=f"{metric_label}_across_all_sessions", chance_line=chance, save_path=fig_path)

    # ── 3. one-sample tests vs chance (per session, BH-FDR) ─────────────
    one_way_df = None
    if chance is not None:
        one_way_results = []
        for s_idx in sessions:
            res = one_sample_vs_chance(metric_df, s_idx,
                                       chance_level=chance,
                                       test=one_way_test,
                                       alternative='two-sided')
            if res is not None:
                one_way_results.append(res)

        if len(one_way_results) > 0:
            one_way_df = pd.DataFrame(one_way_results)
            rejected, p_corr, _, _ = multipletests(one_way_df['p_value'],
                                                    alpha=0.05, method='fdr_bh')
            one_way_df['p_fdr'] = p_corr
            one_way_df['sig_after_correction'] = rejected

            if saving:
                one_way_df.to_csv(
                    os.path.join(metric_path,
                                 f"{metric_label}_vs_null_stats.csv"),
                    index=False)
    else:
        print("chance=None → skipping one-sample tests")

    # ── 4. planned paired comparisons (BH-FDR) ──────────────────────────
    comparison_rows = []
    for subset_key, comp_name, _ in PLANNED_COMPARISONS:
        res = paired_comparison(metric_df, SESSION_SUBSETS[subset_key],
                                test=two_way_test, alternative='two-sided')
        if res is None:
            print(f"  Skipping comparison '{comp_name}': insufficient paired data")
            continue

        # Drop the 'data' key (contains a DataFrame) so we get a clean flat row
        row = {k: v for k, v in res.items() if k != 'data'}
        row['comparison_name'] = comp_name
        comparison_rows.append(row)

    comparisons_df = None
    if len(comparison_rows) > 0:
        comparisons_df = pd.DataFrame(comparison_rows)

        rejected, p_corr, _, _ = multipletests(comparisons_df['p_value'],
                                                alpha=0.05, method='fdr_bh')
        comparisons_df['p_fdr'] = p_corr
        comparisons_df['sig_after_correction'] = rejected

        if saving:
            comparisons_df.to_csv(
                os.path.join(metric_path,
                             f"{metric_label}_paired_comparisons_stats.csv"),
                index=False)

    # ── 5. paired comparison plots ───────────────────────────────────────
    for subset_key, comp_name, title in PLANNED_COMPARISONS:
        fig_save = (os.path.join(metric_path, f"{metric_label}_{comp_name}.svg")
                    if saving else None)
        plot_paired_comparison(metric_df, SESSION_SUBSETS[subset_key],
                               metric_name=metric_label, chance_line=chance,
                               title=title, save_path=fig_save)

    # ── return everything ────────────────────────────────────────────────
    return {
        'metric_df': metric_df,
        'one_way_df': one_way_df,
        'comparisons_df': comparisons_df,
    }

In [ ]:
#toggle saving or not
saving_switch = True
#saving_switch = False

In [ ]:
#partner preference index across the full session
ppi_res = results_compilation_loop(all_behavior_data, metric='partner_preference_index', metric_label='partner_pref_index_full_sesh', res_path=res_path,
                             sessions=None, chance=0,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
ppi_res['metric_df']

In [ ]:
ppi_res['one_way_df']

In [ ]:
ppi_res['comparisons_df']

In [ ]:
#partner preference index across the full session
ppi_first_five_res = results_compilation_loop(all_behavior_data, metric='s1_first_five_min_partner_preference_index', metric_label='partner_pref_index_first_five_min', res_path=res_path,
                             sessions=None, chance=0,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
ppi_first_five_res['metric_df']

In [ ]:
ppi_first_five_res['one_way_df']

In [ ]:
ppi_first_five_res['comparisons_df']

In [ ]:
#partner preference index with chamber time across the full session
ppi_chamber_res = results_compilation_loop(all_behavior_data, metric='partner_preference_index_chambertime', metric_label='partner_pref_index_chamber_time_full_sesh', res_path=res_path,
                             sessions=None, chance=0,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
ppi_chamber_res['metric_df']

In [ ]:
ppi_chamber_res['one_way_df']

In [ ]:
ppi_chamber_res['comparisons_df']

In [ ]:
#partner preference index across the full session
ppi_chamber_first_five_res = results_compilation_loop(all_behavior_data, metric='s1_first_five_min_partner_preference_index_chambertime', metric_label='partner_pref_index_chamber_time_first_five_min', res_path=res_path,
                             sessions=None, chance=0,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
ppi_chamber_first_five_res['metric_df']

In [ ]:
ppi_chamber_first_five_res['one_way_df']

In [ ]:
ppi_chamber_first_five_res['comparisons_df']

In [ ]:
#latency to first partner interaction -- s1 only, seconds
latency_partner_res = results_compilation_loop(all_behavior_data, metric='latency_to_first_partner_int', metric_label='latency_to_first_partner_int', res_path=res_path,
                             sessions=None, chance=None,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
latency_partner_res['metric_df']

In [ ]:
latency_partner_res['comparisons_df']

In [ ]:
#latency to first novel interaction -- s1 only, seconds
latency_novel_res = results_compilation_loop(all_behavior_data, metric='latency_to_first_novel_int', metric_label='latency_to_first_novel_int', res_path=res_path,
                             sessions=None, chance=None,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
latency_novel_res['metric_df']

In [ ]:
latency_novel_res['comparisons_df']

In [ ]:
#total active partner cup investigation, seconds
total_partner_int_time_res = results_compilation_loop(all_behavior_data, metric='total_partner_int_sec', metric_label='total_partner_int_sec', res_path=res_path,
                             sessions=None, chance=None,
                             extractor=derived_extractor(total_partner_int_sec),
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
total_partner_int_time_res['metric_df']

In [ ]:
total_partner_int_time_res['comparisons_df']

In [ ]:
#total active novel cup investigation, seconds
total_novel_int_time_res = results_compilation_loop(all_behavior_data, metric='total_novel_int_sec', metric_label='total_novel_int_sec', res_path=res_path,
                             sessions=None, chance=None,
                             extractor=derived_extractor(total_novel_int_sec),
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
total_novel_int_time_res['metric_df']

In [ ]:
total_novel_int_time_res['comparisons_df']

In [ ]:
#total active cup investigation, both stimuli combined, seconds
total_int_res = results_compilation_loop(all_behavior_data, metric='total_int_sec', metric_label='total_int_sec', res_path=res_path,
                             sessions=None, chance=None,
                             extractor=derived_extractor(total_int_sec),
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
total_int_res['metric_df']

In [ ]:
total_int_res['comparisons_df']

In [ ]:
#number of active partner interaction bouts
n_partner_ints_res = results_compilation_loop(all_behavior_data, metric='n_active_partner_ints', metric_label='n_active_partner_ints', res_path=res_path,
                             sessions=None, chance=None,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
n_partner_ints_res['metric_df']

In [ ]:
n_partner_ints_res['comparisons_df']

In [ ]:
#number of active novel interaction bouts
n_novel_ints_res = results_compilation_loop(all_behavior_data, metric='n_active_novel_ints', metric_label='n_active_novel_ints', res_path=res_path,
                             sessions=None, chance=None,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
n_novel_ints_res['metric_df']

In [ ]:
n_novel_ints_res['comparisons_df']

In [ ]:
#mean partner interaction bout length, seconds
avg_partner_int_len_res = results_compilation_loop(all_behavior_data, metric='avg_active_partner_int_length', metric_label='avg_active_partner_int_length', res_path=res_path,
                             sessions=None, chance=None,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
avg_partner_int_len_res['metric_df']

In [ ]:
avg_partner_int_len_res['comparisons_df']

In [ ]:
#mean novel interaction bout length, seconds
avg_novel_int_len_res = results_compilation_loop(all_behavior_data, metric='avg_active_novel_int_length', metric_label='avg_active_novel_int_length', res_path=res_path,
                             sessions=None, chance=None,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
avg_novel_int_len_res['metric_df']

In [ ]:
avg_novel_int_len_res['comparisons_df']

In [ ]:
#number of partner<->novel bout switches
n_switches_res = results_compilation_loop(all_behavior_data, metric='num_int_type_switches', metric_label='num_int_type_switches', res_path=res_path,
                             sessions=None, chance=None,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
n_switches_res['metric_df']

In [ ]:
n_switches_res['comparisons_df']

In [ ]:
#proportion of partner-chamber frames spent actively investigating
prop_pchamb_active_res = results_compilation_loop(all_behavior_data, metric='proportion_partner_chamber_frames_active', metric_label='proportion_partner_chamber_frames_active', res_path=res_path,
                             sessions=None, chance=None,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
prop_pchamb_active_res['metric_df']

In [ ]:
prop_pchamb_active_res['comparisons_df']

In [ ]:
#proportion of novel-chamber frames spent actively investigating
prop_nchamb_active_res = results_compilation_loop(all_behavior_data, metric='proportion_novel_chamber_frames_active', metric_label='proportion_novel_chamber_frames_active', res_path=res_path,
                             sessions=None, chance=None,
                             one_way_test='ttest', two_way_test='wilcoxon',
                             saving=saving_switch)

In [ ]:
prop_nchamb_active_res['metric_df']

In [ ]:
prop_nchamb_active_res['comparisons_df']

#### Compiled statistics across additional metrics

One table of the five planned cross-session comparisons for every metric, and one of
the vs-chance tests where a chance level applies. BH-FDR is applied within each
metric across its five comparisons

In [ ]:
#compile every metric's planned-comparison stats into one reviewable table
addl_metric_results = {
    'latency_to_first_partner_int': latency_partner_res,
    'latency_to_first_novel_int': latency_novel_res,
    'total_partner_int_sec': total_partner_int_time_res,
    'total_novel_int_sec': total_novel_int_time_res,
    'total_int_sec': total_int_res,
    'n_active_partner_ints': n_partner_ints_res,
    'n_active_novel_ints': n_novel_ints_res,
    'avg_active_partner_int_length': avg_partner_int_len_res,
    'avg_active_novel_int_length': avg_novel_int_len_res,
    'num_int_type_switches': n_switches_res,
    'proportion_partner_chamber_frames_active': prop_pchamb_active_res,
    'proportion_novel_chamber_frames_active': prop_nchamb_active_res,
}

all_comparisons = pd.concat(
    [r['comparisons_df'].assign(metric=label)
     for label, r in addl_metric_results.items() if r['comparisons_df'] is not None],
    ignore_index=True)

all_one_way = pd.concat(
    [r['one_way_df'].assign(metric=label)
     for label, r in addl_metric_results.items() if r['one_way_df'] is not None],
    ignore_index=True) if any(r['one_way_df'] is not None for r in addl_metric_results.values()) else None

all_comparisons.to_csv(os.path.join(res_path, 'additional_behavior_metrics_paired_comparisons.csv'), index=False)
if all_one_way is not None:
    all_one_way.to_csv(os.path.join(res_path, 'additional_behavior_metrics_vs_null.csv'), index=False)

print(f'{len(addl_metric_results)} metrics | {len(all_comparisons)} planned comparisons')
all_comparisons

#### Within-session partner vs. novel comparisons

`results_compilation_loop` compares each metric **across sessions**. It cannot answer
"did the animal approach the partner sooner than the novel *within* a given session?"

`compare_metrics_within_sessions` does the paired partner-vs-novel test per session,
with BH-FDR applied across the five sessions.

In [ ]:
#paired partner vs. novel within each session, for the three partner/novel metric pairs
within_session_pairs = [
    ('latency_to_first_int', latency_partner_res, latency_novel_res),
    ('total_int_sec_p_vs_n', total_partner_int_time_res, total_novel_int_time_res),
    ('avg_int_bout_length', avg_partner_int_len_res, avg_novel_int_len_res),
]

within_session_results = {}
for name, p_res, n_res in within_session_pairs:
    out = compare_metrics_within_sessions(p_res['metric_df'], n_res['metric_df'],
                                          label_a='partner', label_b='novel',
                                          test='wilcoxon', alternative='two-sided')
    within_session_results[name] = out
    print(f'\n=== {name} ===')
    print(out[['short_label', 'n', 'mean_partner', 'mean_novel',
               'p_uncorrected', 'p_fdr', 'significant']].to_string(index=False))

within_session_df = pd.concat(
    [df.assign(metric=name) for name, df in within_session_results.items()],
    ignore_index=True)
within_session_df.to_csv(
    os.path.join(res_path, 'additional_behavior_metrics_within_session_partner_vs_novel.csv'),
    index=False)
within_session_df

#### Partner vs. novel bar plots

Grouped bars per session, individual animals paired by gray lines, SEM error
bars, x-axis labelled by day.

Bar fill marks where a familiar partner actually exists: the `Partner` bar takes the
session colour only for `SESSION_SUBSETS['familiar']` (sessions 2, 4, 5 -- days 2, 9, 23) and
stays gray on the novel/novel sessions (days 1, 8), where the "partner" is merely the animal
that will become familiar. Pass `color_familiar_only=False` to color every session.

Animals are paired with the same inner merge `compare_metrics_within_sessions` uses, so the n
plotted always matches the n tested. `ylim` is left unset by default so no data point is
cropped -- note that one animal's day-1 partner latency (284.5 s, 8020) is ~3x the largest
mean and compresses the bars; pass e.g. `ylim=(0, 160)` if a tighter axis reads better.

The function takes any two `extract_metric`-style frames.

In [ ]:
from matplotlib.patches import Patch

NEUTRAL_GRAY = '#C8C8C8'


def plot_paired_bars_across_sessions(df_a, df_b, label_a='Partner', label_b='Novel',
                                     session_indices=None, metric_name='',
                                     color_familiar_only=True, bar_width=0.38,
                                     figsize=(5, 4.5), title=None, ylim=None,
                                     save_path=None):
    """
    Grouped bar plot of two paired metrics within each session, with individual animal lines.

    df_b is drawn on the left in gray (the reference stimulus); df_a on the right, filled with
    the session colour. Animals are paired within a session by an inner merge, so the n plotted matches the n tested.

    Parameters
    ----------
    df_a, df_b : pd.DataFrame
        Two extract_metric()-style frames (columns animal, session_idx, value, ...).
        df_a is the focal/coloured metric, df_b the gray reference.
    label_a, label_b : str
        Legend labels; also used as the merge suffixes.
    session_indices : list[int] or None
        Sessions to plot (default: sessions present in both frames, ascending).
    metric_name : str
        Y-axis label.
    color_familiar_only : bool
        If True, fill the df_a bar with the session colour only for sessions in
        SESSION_SUBSETS['familiar'] -- the post-cohousing sessions, where a familiar partner
        actually exists. Other sessions get the same gray as df_b.
    bar_width : float
        Width of each bar; the two bars in a group sit at +/- bar_width/2 from the centre.
    ylim : tuple or None
        Passed to ax.set_ylim. Left unset by default so no data point is cropped.
    save_path : str or None

    Returns
    -------
    (fig, ax)
    """
    if session_indices is None:
        session_indices = sorted(
            set(df_a['session_idx'].unique()) & set(df_b['session_idx'].unique())
        )

    va, vb = f'value_{label_a}', f'value_{label_b}'
    merged = df_a[['animal', 'session_idx', 'value']].merge(
        df_b[['animal', 'session_idx', 'value']],
        on=['animal', 'session_idx'], suffixes=(f'_{label_a}', f'_{label_b}')
    )

    fig, ax = plt.subplots(figsize=figsize)
    centers = np.arange(len(session_indices))
    half = bar_width / 2

    for center, s_idx in zip(centers, session_indices):
        s_data = merged[merged['session_idx'] == s_idx].dropna(subset=[va, vb])
        if len(s_data) == 0:
            continue

        x_b, x_a = center - half, center + half
        color_a = (SESSION_CONFIG['colors'][s_idx]
                   if (not color_familiar_only or s_idx in SESSION_SUBSETS['familiar'])
                   else NEUTRAL_GRAY)

        # bars = means
        ax.bar(x_b, s_data[vb].mean(), width=bar_width, color=NEUTRAL_GRAY, zorder=1)
        ax.bar(x_a, s_data[va].mean(), width=bar_width, color=color_a, zorder=1)

        # one line per animal, df_b value -> df_a value
        for _, row in s_data.iterrows():
            ax.plot([x_b, x_a], [row[vb], row[va]],
                    color='gray', alpha=0.5, linewidth=0.8, zorder=5)

        # SEM error bars
        ax.errorbar([x_b, x_a], [s_data[vb].mean(), s_data[va].mean()],
                    yerr=[s_data[vb].sem(), s_data[va].sem()],
                    fmt='none', ecolor='black', capsize=4, capthick=1.5,
                    elinewidth=1.5, zorder=10)

    # x-axis: day number, tick label tinted by session (same idiom as plot_metric_across_sessions)
    ax.set_xticks(centers)
    ax.set_xticklabels([SESSION_CONFIG['days'][s] for s in session_indices], fontsize=12)
    for ticklabel, s_idx in zip(ax.get_xticklabels(), session_indices):
        ticklabel.set_color(SESSION_CONFIG['colors'][s_idx])
    ax.set_xlabel('Day', fontsize=13)

    if metric_name:
        ax.set_ylabel(metric_name, fontsize=13)
    if ylim is not None:
        ax.set_ylim(ylim)
    if title is not None:
        ax.set_title(title)

    ax.legend(handles=[Patch(facecolor=NEUTRAL_GRAY, label=label_b),
                       Patch(facecolor=SESSION_CONFIG['colors'][4], label=label_a)],
              frameon=False, fontsize=12, loc='upper right')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()
    return fig, ax

In [ ]:
saving_switch

In [ ]:
#latency to first interaction, partner vs. novel
fig, ax = plot_paired_bars_across_sessions(
    latency_partner_res['metric_df'], latency_novel_res['metric_df'],
    label_a='Partner', label_b='Novel',
    metric_name='Latency to first interaction (s)',
    save_path=os.path.join(res_path, 'latency_partner_vs_novel_bars.svg') if saving_switch else None)

In [ ]:
latency_partner_res['metric_df']

In [ ]:
latency_stats = compare_metrics_within_sessions(
    latency_partner_res['metric_df'], latency_novel_res['metric_df'],
    label_a='partner', label_b='novel',
    test='wilcoxon', alternative='two-sided')


if saving_switch:
    latency_stats.to_csv(
        os.path.join(res_path, 'latency_to_first_partner_vs_novel_int_stats.csv'), index=False)
latency_stats

In [ ]:
#total interaction time, partner vs. novel
fig, ax = plot_paired_bars_across_sessions(
    total_partner_int_time_res['metric_df'], total_novel_int_time_res['metric_df'],
    label_a='Partner', label_b='Novel',
    metric_name='Total interaction time (s)',
    save_path=os.path.join(res_path, 'total_int_time_partner_vs_novel_bars.svg') if saving_switch else None)

In [ ]:
#average interaction length, partner vs. novel
fig, ax = plot_paired_bars_across_sessions(
    avg_partner_int_len_res['metric_df'], avg_novel_int_len_res['metric_df'],
    label_a='Partner', label_b='Novel',
    metric_name='Average interaction bout length (s)',
    save_path=os.path.join(res_path, 'avg_int_length_partner_vs_novel_bars.svg') if saving_switch else None)

#### Seconds spent actively investigating partner vs. novel

First five minutes of s1 -- seconds investigating each stimulus.: `n_ints * avg_int_length` 


In [ ]:
#seconds of the first five minutes of s1 spent actively investigating each stimulus
#note the _sec suffix on the variable names -- the bare names are the extractor functions
s1_first5_partner_sec = derived_extractor(s1_first5_sec_partner)(
    all_behavior_data, 's1_first5_sec_partner',
    session_indices=SESSION_SUBSETS['social'], animals=None)
s1_first5_novel_sec = derived_extractor(s1_first5_sec_novel)(
    all_behavior_data, 's1_first5_sec_novel',
    session_indices=SESSION_SUBSETS['social'], animals=None)

s1_first5_partner_sec

In [ ]:
s1_first5_novel_sec

In [ ]:
both_animals_s1_first5_int_sec = pd.concat([s1_first5_novel_sec,s1_first5_partner_sec]).reset_index(drop=True)

if saving_switch:
    both_animals_s1_first5_int_sec.to_csv(
        os.path.join(res_path, 's1_first_five_min_seconds_partner_vs_novel_interactions_summary.csv'),
        index=False)

both_animals_s1_first5_int_sec

In [ ]:
fig, ax = plot_paired_bars_across_sessions(
    s1_first5_partner_sec, s1_first5_novel_sec,
    label_a='Partner', label_b='Novel',
    metric_name='Time investigating (s)',
    save_path=(os.path.join(res_path,
                            's1_first_five_min_seconds_partner_vs_novel_bars.svg')
               if saving_switch else None))

s1_first5_sec_stats = compare_metrics_within_sessions(
    s1_first5_partner_sec, s1_first5_novel_sec,
    label_a='partner', label_b='novel',
    test='wilcoxon', alternative='two-sided')
if saving_switch:
    s1_first5_sec_stats.to_csv(
        os.path.join(res_path, 's1_first_five_min_seconds_partner_vs_novel_stats.csv'),
        index=False)
s1_first5_sec_stats

#### Full 3-hr partner preference tests -- seconds at each cup

Seconds at the cups were never recorded directly, and different cameras were used across
tests because of necessary hardware changes. They are reconstructed in a separate workbook,
`good_imaging_animal_ppt_scores_seconds.xlsx`, as

```
total_{stimulus}_cup_seconds = (num_tracked_frames_total * percent_{stimulus}_cup_frames) / framerate
```

with a per-test framerate.

In [ ]:
#full 3-hr partner preference tests: reconstructed seconds at each cup
#long-format workbook (one row per animal-session); note the lowercase 'animal' column
PPT_SECONDS_PATH = r"C:\Users\sheer\UCB-O365\Zoe Donaldson - Sheeran_vHC_imaging\Figures\manuscript_figures\miniscope_imaging_figure\imaging_animal_ppt_results\good_imaging_animal_ppt_scores_seconds.xlsx"

ppt_seconds_raw = pd.read_excel(PPT_SECONDS_PATH, sheet_name=0)
ppt_seconds_raw['animal'] = ppt_seconds_raw['animal'].astype(str)
ppt_seconds_raw = ppt_seconds_raw[
    ppt_seconds_raw['animal'].isin(ANIMAL_PATHS.keys())].reset_index(drop=True)
print(f"{ppt_seconds_raw['animal'].nunique()} of {len(ANIMAL_PATHS)} imaging animals "
      f"| {len(ppt_seconds_raw)} animal-sessions in the PPT seconds workbook")

ppt_seconds_raw

In [ ]:
ppt_partner_sec = make_ppt_seconds_frame(ppt_seconds_raw, 'partner')
ppt_novel_sec = make_ppt_seconds_frame(ppt_seconds_raw, 'novel')

ppt_novel_sec

In [ ]:
both_animals_ppt_cup_sec = pd.concat([ppt_novel_sec,ppt_partner_sec]).reset_index(drop=True)

if saving_switch:
    both_animals_ppt_cup_sec.to_csv(
        os.path.join(res_path, 'ppt_novel_partner_seconds_at_cup_summary.csv'),
        index=False)

both_animals_ppt_cup_sec

In [ ]:
fig, ax = plot_paired_bars_across_sessions(
    ppt_partner_sec, ppt_novel_sec,
    label_a='Partner', label_b='Novel',
    metric_name='Time at cup (s)',
    save_path=(os.path.join(res_path, 'ppt_cup_seconds_partner_vs_novel_bars.svg')
               if saving_switch else None))

ppt_cup_sec_stats = compare_metrics_within_sessions(
    ppt_partner_sec, ppt_novel_sec,
    label_a='partner', label_b='novel',
    test='wilcoxon', alternative='two-sided')
if saving_switch:
    ppt_cup_sec_stats.to_csv(
        os.path.join(res_path, 'ppt_cup_seconds_partner_vs_novel_stats.csv'), index=False)
ppt_cup_sec_stats

#### Paired bar plots for 3 hr PPT cup time and first five minutes imaging session interaction time in proportion of time rather than seconds

First five minutes of s1 -- proportion of total time investigating each stimulus

In [ ]:
#proportion of the first five minutes of s1 spent actively investigating each stimulus
#derived from the stored bout count x mean bout length
s1_first5_partner = derived_extractor(s1_first5_prop_time_partner)(
    all_behavior_data, 's1_first5_prop_time_partner',
    session_indices=SESSION_SUBSETS['social'], animals=None)
s1_first5_novel = derived_extractor(s1_first5_prop_time_novel)(
    all_behavior_data, 's1_first5_prop_time_novel',
    session_indices=SESSION_SUBSETS['social'], animals=None)

fig, ax = plot_paired_bars_across_sessions(
    s1_first5_partner, s1_first5_novel,
    label_a='Partner', label_b='Novel',
    metric_name='Prop. total time',
    save_path=(os.path.join(res_path,
                            's1_first_five_min_prop_time_partner_vs_novel_bars.svg')
               if saving_switch else None))

s1_first5_stats = compare_metrics_within_sessions(
    s1_first5_partner, s1_first5_novel,
    label_a='partner', label_b='novel',
    test='wilcoxon', alternative='two-sided')
if saving_switch:
    s1_first5_stats.to_csv(
        os.path.join(res_path, 's1_first_five_min_prop_time_partner_vs_novel_stats.csv'),
        index=False)
s1_first5_stats

#### Full 3-hr partner preference tests -- proportion of total time at each cup

Read from `imaging_animal_ppt_scores.xlsx`, columns `{prefix}_{stimulus}_cup` with the prefixes
already in `session_config.PPT_SESSION_MAP` (`SS` -> 2, `OS_presep` -> 4, `OS_postsep` -> 5).
These are ezTrack outputs: the fraction of recorded test frames the animal was at that cup, on
0-1. `make_ppt_cup_time_frame` reshapes them into the same tidy schema `extract_metric`
produces, which is what lets PPT data share the plotting and stats helpers with everything else
in this notebook.

Distinct from `make_ppt_lookup`, which returns the preference score
`partner / (partner + novel)`; these are the two underlying quantities that score is built from.

In [ ]:
#full 3-hr partner preference tests: proportion of total frames spent at each cup
PPT_SCORES_PATH = r"C:\Users\sheer\UCB-O365\Zoe Donaldson - Sheeran_vHC_imaging\Figures\manuscript_figures\miniscope_imaging_figure\imaging_animal_ppt_results\imaging_animal_ppt_scores.xlsx"

ppt_scores_raw = pd.read_excel(PPT_SCORES_PATH, sheet_name=0)
ppt_scores_raw['Animal'] = ppt_scores_raw['Animal'].astype(str)
ppt_scores_raw = ppt_scores_raw[
    ppt_scores_raw['Animal'].isin(ANIMAL_PATHS.keys())].reset_index(drop=True)
print(f"{len(ppt_scores_raw)} of {len(ANIMAL_PATHS)} imaging animals found in the PPT workbook")

ppt_partner_cup = make_ppt_cup_time_frame(ppt_scores_raw, 'partner')
ppt_novel_cup = make_ppt_cup_time_frame(ppt_scores_raw, 'novel')

fig, ax = plot_paired_bars_across_sessions(
    ppt_partner_cup, ppt_novel_cup,
    label_a='Partner', label_b='Novel',
    metric_name='Prop. total time',
    save_path=(os.path.join(res_path, 'ppt_cup_time_partner_vs_novel_bars.svg')
               if saving_switch else None))

ppt_cup_stats = compare_metrics_within_sessions(
    ppt_partner_cup, ppt_novel_cup,
    label_a='partner', label_b='novel',
    test='wilcoxon', alternative='two-sided')
if saving_switch:
    ppt_cup_stats.to_csv(
        os.path.join(res_path, 'ppt_cup_time_partner_vs_novel_stats.csv'), index=False)
ppt_cup_stats